In [1]:
!pip install -q pandas pillow numpy matplotlib


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import math
import os
import pandas as pd
import numpy as np
from PIL import Image, ImageChops
import matplotlib.pyplot as plt

In [3]:
TILE_SIZE = 512

# Example token assumptions for study purposes
LOW_DETAIL_BASE_TOKENS = 85          # coarse single-pass
HIGH_DETAIL_BASE_TOKENS = 85         # base overhead
HIGH_DETAIL_PER_TILE = 170          # per 512x512 tile (illustrative)

print("Tile Size:", TILE_SIZE)

Tile Size: 512


In [4]:
def tiles_needed(width, height, tile_size=TILE_SIZE):
    cols = math.ceil(width / tile_size)
    rows = math.ceil(height / tile_size)
    return rows, cols, rows * cols

def estimate_tokens(width, height, detail="high"):
    rows, cols, tiles = tiles_needed(width, height)
    
    if detail == "low":
        # low detail = single coarse representation
        tokens = LOW_DETAIL_BASE_TOKENS
        
    elif detail == "high":
        tokens = HIGH_DETAIL_BASE_TOKENS + tiles * HIGH_DETAIL_PER_TILE
        
    else:
        raise ValueError("detail must be low or high")
    
    return {
        "width": width,
        "height": height,
        "rows": rows,
        "cols": cols,
        "tiles": tiles,
        "detail": detail,
        "tokens": tokens
    }

In [5]:
images = {
    "Thumbnail": (320, 240),
    "1080p": (1920, 1080),
    "4K": (3840, 2160),
    "Large Example": (4000, 3000)
}

rows = []

for name, (w, h) in images.items():
    for detail in ["low", "high"]:
        result = estimate_tokens(w, h, detail)
        result["image_type"] = name
        rows.append(result)

df = pd.DataFrame(rows)
df

,width,height,rows,cols,tiles,detail,tokens,image_type
0,320,240,1,1,1,low,85,Thumbnail
1,320,240,1,1,1,high,255,Thumbnail
2,1920,1080,3,4,12,low,85,1080p
3,1920,1080,3,4,12,high,2125,1080p
4,3840,2160,5,8,40,low,85,4K
5,3840,2160,5,8,40,high,6885,4K
6,4000,3000,6,8,48,low,85,Large Example
7,4000,3000,6,8,48,high,8245,Large Example


In [6]:
summary = df[[
    "image_type",
    "width",
    "height",
    "detail",
    "rows",
    "cols",
    "tiles",
    "tokens"
]]

summary

,image_type,width,height,detail,rows,cols,tiles,tokens
0,Thumbnail,320,240,low,1,1,1,85
1,Thumbnail,320,240,high,1,1,1,255
2,1080p,1920,1080,low,3,4,12,85
3,1080p,1920,1080,high,3,4,12,2125
4,4K,3840,2160,low,5,8,40,85
5,4K,3840,2160,high,5,8,40,6885
6,Large Example,4000,3000,low,6,8,48,85
7,Large Example,4000,3000,high,6,8,48,8245


In [ ]:
high_df = df[df["detail"] == "high"].copy()

print("Estimated High Detail Tokens by Image Size\n")

for _, row in high_df.iterrows():
    image_type = row["image_type"]
    tokens = int(row["tokens"])

    bars = "█" * max(1, tokens // 300)

    print(f"{image_type:15} {bars} ({tokens} tokens)")
    

Estimated High Detail Tokens by Image Size

Thumbnail       █ (255 tokens)
1080p           ███████ (2125 tokens)
4K              ██████████████████████ (6885 tokens)
Large Example   ███████████████████████████ (8245 tokens)


In [17]:
high_df[["image_type", "width", "height", "tiles", "tokens"]]

,image_type,width,height,tiles,tokens
1,Thumbnail,320,240,1,255
3,1080p,1920,1080,12,2125
5,4K,3840,2160,40,6885
7,Large Example,4000,3000,48,8245


In [18]:
summary = high_df[["image_type", "width", "height", "tiles", "tokens"]]

print(summary.to_string(index=False))

   image_type  width  height  tiles  tokens
    Thumbnail    320     240      1     255
        1080p   1920    1080     12    2125
           4K   3840    2160     40    6885
Large Example   4000    3000     48    8245


In [10]:
large = estimate_tokens(4000, 3000, "high")
large

{'width': 4000,
 'height': 3000,
 'rows': 6,
 'cols': 8,
 'tiles': 48,
 'detail': 'high',
 'tokens': 8245}

For a 4000×3000 image:

- Columns = ceil(4000 / 512)
- Rows = ceil(3000 / 512)
- Total tiles = rows × cols

Estimated tokens:

base_tokens + (tiles × per_tile_tokens)

This demonstrates why very large images become expensive quickly.

In [11]:
def resize_keep_aspect(width, height, max_side=2048):
    scale = min(max_side / width, max_side / height, 1.0)
    return int(width * scale), int(height * scale), scale

resize_keep_aspect(4000, 3000)

(2048, 1536, 0.512)

Common preprocessing strategy:

1. Preserve aspect ratio
2. Shrink largest side to a cap (example: 2048 px)
3. Tile resized image into 512×512 regions

This reduces unnecessary tiles while keeping usable detail.

In [12]:
orig = estimate_tokens(4000, 3000, "high")

rw, rh, scale = resize_keep_aspect(4000, 3000, max_side=2048)
resized = estimate_tokens(rw, rh, "high")

pd.DataFrame([orig, resized])[[
    "width","height","tiles","tokens"
]]

,width,height,tiles,tokens
0,4000,3000,48,8245
1,2048,1536,12,2125


In [13]:
tips = pd.DataFrame({
    "Technique": [
        "Crop whitespace",
        "Resize oversized image",
        "Split multipage docs",
        "Use detail=low first pass",
        "Deskew + crop margins",
        "Send ROI only (table region)"
    ],
    "Impact": [
        "Fewer tiles",
        "Fewer tiles",
        "Process only needed pages",
        "Cheap triage mode",
        "Cleaner OCR + fewer tiles",
        "Avoid paying for irrelevant pixels"
    ]
})

tips

,Technique,Impact
0,Crop whitespace,Fewer tiles
1,Resize oversized image,Fewer tiles
2,Split multipage docs,Process only needed pages
3,Use detail=low first pass,Cheap triage mode
4,Deskew + crop margins,Cleaner OCR + fewer tiles
5,Send ROI only (table region),Avoid paying for irrelevant pixels


In [14]:
def trim_whitespace(img):
    bg = Image.new(img.mode, img.size, img.getpixel((0,0)))
    diff = ImageChops.difference(img, bg)
    bbox = diff.getbbox()
    return img.crop(bbox) if bbox else img

# synthetic demo image
img = Image.new("RGB", (1600, 1200), "white")
cropped = trim_whitespace(img)

print("Original:", img.size)
print("Cropped:", cropped.size)

Original: (1600, 1200)
Cropped: (1600, 1200)


In [15]:
COST_PER_1M_TOKENS = 0.15   # example placeholder

monthly_images = 100000

tokens_per_image = estimate_tokens(1920,1080,"high")["tokens"]

monthly_tokens = monthly_images * tokens_per_image
monthly_cost = (monthly_tokens / 1_000_000) * COST_PER_1M_TOKENS

print("Tokens/Image:", tokens_per_image)
print("Monthly Tokens:", monthly_tokens)
print("Estimated Monthly Cost:", round(monthly_cost,2))

Tokens/Image: 2125
Monthly Tokens: 212500000
Estimated Monthly Cost: 31.88


In [16]:
strategy = pd.DataFrame({
    "Scenario": [
        "Preview scan",
        "Invoice OCR",
        "Dense engineering drawing",
        "Large scanned page",
        "Unknown input"
    ],
    "Recommended Mode": [
        "detail=low",
        "detail=high cropped ROI",
        "detail=high",
        "resize then high",
        "low then escalate"
    ]
})

strategy

,Scenario,Recommended Mode
0,Preview scan,detail=low
1,Invoice OCR,detail=high cropped ROI
2,Dense engineering drawing,detail=high
3,Large scanned page,resize then high
4,Unknown input,low then escalate


## Final Answers

### 1. How are tokens calculated for a 4000×3000 image?

The image is divided into 512×512 tiles after optional resizing. Total token usage grows with tile count. Large images can require many tiles, significantly increasing cost.

### 2. How is the image resized before tiling?

A common strategy is aspect-ratio-preserving resize using a maximum side cap (example: 2048 px), then tile the resized image.

### 3. What preprocessing reduces unnecessary token usage?

- Remove whitespace
- Crop margins
- Resize oversized images
- Send only relevant regions
- Use low detail triage first
- Split multipage inputs

## Conclusion

Vision cost is driven more by image area than file size. High-detail processing scales with tile count, so large images should be cropped, resized, or routed selectively.

Recommended production pipeline:

1. Low-detail preview
2. Detect relevant region
3. High-detail only where needed
4. Escalate premium models selectively